In [ ]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories

target_similarity=defaultdict(list)
clause_weight_threshold = 0
number_of_examples = 1
clause_drop_p = 0.0
factor = 20
clauses = 80
T = factor*40
s = 5.0
accumulation = 1
epochs = 25

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)
            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)
            available_pair_list = []
            pairs_output_active = []
            for pair, score in pair_list:
                word1, word2 = pair[0], pair[1]
                if all(word in target_words for word in [word1, word2]):
                    available_pair_list.append([pair,score])
                    pairs_output_active.append([vectorizer_X.vocabulary_[word1], vectorizer_X.vocabulary_[word2]])
            
            result_filepath = Dicrectories.test(dataset_name,"cross_phase2")
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("Accumulation: %d" % accumulation)
                print("Examples: %d" % number_of_examples)
                print("No of features: %d" % number_of_features)
                print("Clauses: %d\n" % clauses)
                
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    print("\nEpoch #: %d" % e)
                    epoch_time = 0
                    for pair_index, pair in enumerate(pairs_output_active):
                        pair_output_active = np.empty(2, dtype=np.uint32)
                        pair_output_active[0] = pair[0]
                        pair_output_active[1] = pair[1]
                        start_training = time()
                        tm = TMAutoEncoder(clauses, T, s, pair_output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)
                        tm.knowledge_fit(
                            number_of_examples = number_of_examples,
                            number_of_features = number_of_features,
                            )
                        stop_training = time()
                        pair_time = stop_training - start_training
                        epoch_time = epoch_time + pair_time

                        profile = np.empty((2, clauses))
                        weights = tm.get_weights(0)
                        profile[0,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                        weights = tm.get_weights(1)
                        profile[1,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                        similarity = cosine_similarity(profile)

                        sorted_index = np.argsort(-1*similarity[0,:])
                        target_similarity[available_pair_list[pair_index][0]]  = similarity[0,sorted_index[1]]

                    Tools.print_training_time(epoch_time)
                    total_training = total_training + epoch_time
                    eval.calculate(target_similarity,available_pair_list)
                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()
                Tools.print_training_time(total_training)

In [3]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories

target_similarity=defaultdict(list)
clause_weight_threshold = 0
number_of_examples = 1
clause_drop_p = 0.0
factor = 20
clauses = 80
T = factor*40
s = 5.0
accumulation = 24
epochs = 25

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)
            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)
            available_pair_list = []
            pairs_output_active = []
            base = 0
            for pair, score in pair_list:
                word1, word2 = pair[0], pair[1]
                if score > base:
                    base = score
                if all(word in target_words for word in [word1, word2]):
                    available_pair_list.append([pair,score])
                    pairs_output_active.append([vectorizer_X.vocabulary_[word1], vectorizer_X.vocabulary_[word2]])
            print(base)

3.94
